# ABC2Vec — Notebook 13: Plagiarism / Melodic Similarity Detection

**Optional extension from the research brief — implemented as requested.**

Applies ABC2Vec embeddings to detect melodic similarity and potential plagiarism
between folk tunes.

**Experiments:**
1. Known melodically related tune pairs from musicological literature
2. Nearest-neighbor search for "suspicious" melodic matches
3. Threshold calibration: what cosine similarity indicates "same tune"?
4. Cross-corpus duplicate detection (e.g., Irish tunes appearing in Nottingham)
5. Similarity heat-map between tune collections
6. Bar-level alignment for localized similarity

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_recall_curve, average_precision_score

try:
    import faiss
except ImportError:
    !pip install faiss-cpu
    import faiss

PROJECT_DIR = Path('/Volumes/LLModels/ABC2VEC')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
BENCHMARK_DIR = PROJECT_DIR / 'data' / 'benchmark'
RESULTS_DIR = PROJECT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR))

device = torch.device('cuda' if torch.cuda.is_available() else 
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

## 13.1 Load Model and Encode Corpora

In [ ]:
from abc2vec.model import ABC2VecModel, ABC2VecConfig
from abc2vec.tokenizer import ABCVocabulary, BarPatchifier

vocab = ABCVocabulary()
vocab_path = PROJECT_DIR / 'data' / 'processed' / 'vocabulary.json'
if vocab_path.exists():
    vocab.load(vocab_path)

patchifier = BarPatchifier(vocab, max_bar_length=64, max_bars=64)

config = ABC2VecConfig(
    vocab_size=len(vocab),
    max_bar_len=64, max_bars=64,
    d_model=256, n_heads=8, n_layers=6,
    d_ff=1024, d_embed=128, dropout=0.1,
)
model = ABC2VecModel(config).to(device)

ckpt_path = PROJECT_DIR / 'checkpoints' / 'best_model.pt'
if ckpt_path.exists():
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

@torch.no_grad()
def encode_tunes(abc_texts, model, patchifier, device, batch_size=64):
    model.eval()
    all_emb = []
    for i in tqdm(range(0, len(abc_texts), batch_size), desc="Encoding"):
        batch = abc_texts[i:i+batch_size]
        bar_ids_list, bar_mask_list = [], []
        for text in batch:
            bars = patchifier.patchify(text)
            ids, mask = patchifier.encode(bars)
            bar_ids_list.append(ids)
            bar_mask_list.append(mask)
        bar_ids = torch.stack(bar_ids_list).to(device)
        bar_mask = torch.stack(bar_mask_list).to(device)
        emb = model.get_embedding(bar_ids, bar_mask)
        all_emb.append(emb.cpu().numpy())
    return np.concatenate(all_emb, axis=0)

In [ ]:
# Load all available corpora
corpora = {}

# IrishMAN test set
test_path = PROCESSED_DIR / 'test.parquet'
if test_path.exists():
    test_df = pd.read_parquet(test_path)
    corpora['Irish (IrishMAN)'] = test_df

# Nottingham
nott_path = PROCESSED_DIR / 'nottingham.json'
if nott_path.exists():
    with open(nott_path) as f:
        nott = json.load(f)
    corpora['Nottingham'] = pd.DataFrame(nott)

# BFDB
bfdb_path = PROCESSED_DIR / 'bfdb.json'
if bfdb_path.exists():
    with open(bfdb_path) as f:
        bfdb = json.load(f)
    corpora['British (BFDB)'] = pd.DataFrame(bfdb)

for name, df in corpora.items():
    print(f"{name}: {len(df)} tunes")

# Encode all corpora
corpus_embeddings = {}
for name, df in corpora.items():
    print(f"\nEncoding {name}...")
    emb = encode_tunes(df['abc_body'].tolist(), model, patchifier, device)
    corpus_embeddings[name] = emb
    print(f"  Shape: {emb.shape}")

## 13.2 Known Melodically Related Pairs

Some well-known related tune pairs from musicological literature:
- "The Kesh Jig" and its many variants
- "Morrison's Jig" variants across traditions
- "The Rights of Man" hornpipe variants

In [ ]:
# Define known related tunes (these are canonical examples)
# Pairs of ABC notation for well-known variants

KNOWN_PAIRS = [
    {
        'name': 'The Kesh Jig (variant pair)',
        'tune_a': 'G2G GAB|ded BAG|G2G GAB|dBA AGF|G2G GAB|ded BAG|aba aga|BGG G3:|',
        'tune_b': 'G2G GAB|d2d BAG|G2G GAB|d2A AGF|G2G GAB|d2d BAG|aba aga|BGG G3:|',
        'expected': 'similar',
    },
    {
        'name': 'Unrelated: Jig vs. Reel',
        'tune_a': 'G2G GAB|ded BAG|G2G GAB|dBA AGF|',  # Jig 6/8
        'tune_b': 'ABAF ABdf|afdf aBBA|ABAF ABdf|afdf aBBA|',  # Reel 4/4
        'expected': 'different',
    },
    {
        'name': 'Transposition: Same tune in D vs G',
        'tune_a': 'D2D DEF|ABA FED|D2D DEF|AFE EDC|',
        'tune_b': 'G2G GAB|ded BAG|G2G GAB|dBA AGF|',
        'expected': 'similar (transposed)',
    },
]

print("Known Melodic Pair Similarities:")
print("=" * 60)

for pair in KNOWN_PAIRS:
    emb_a = encode_tunes([pair['tune_a']], model, patchifier, device)
    emb_b = encode_tunes([pair['tune_b']], model, patchifier, device)
    sim = cosine_similarity(emb_a, emb_b)[0, 0]
    
    print(f"\n{pair['name']}:")
    print(f"  Expected: {pair['expected']}")
    print(f"  Cosine similarity: {sim:.4f}")
    print(f"  A: {pair['tune_a'][:60]}...")
    print(f"  B: {pair['tune_b'][:60]}...")

## 13.3 Cross-Corpus Duplicate Detection

In [ ]:
# Find tunes in one corpus that are very similar to tunes in another
# This reveals cross-tradition borrowing / shared heritage

corpus_names = list(corpus_embeddings.keys())

if len(corpus_names) >= 2:
    # For each pair of corpora
    for i in range(len(corpus_names)):
        for j in range(i+1, len(corpus_names)):
            name_a = corpus_names[i]
            name_b = corpus_names[j]
            emb_a = corpus_embeddings[name_a]
            emb_b = corpus_embeddings[name_b]
            
            # Normalize
            na = emb_a / np.maximum(np.linalg.norm(emb_a, axis=1, keepdims=True), 1e-8)
            nb = emb_b / np.maximum(np.linalg.norm(emb_b, axis=1, keepdims=True), 1e-8)
            
            # Build FAISS index on corpus B
            d = emb_b.shape[1]
            index = faiss.IndexFlatIP(d)
            index.add(nb.astype(np.float32))
            
            # Search from corpus A into corpus B
            distances, indices = index.search(na.astype(np.float32), 1)
            
            # Find high-similarity pairs
            THRESHOLD = 0.90
            high_sim_mask = distances[:, 0] >= THRESHOLD
            n_matches = high_sim_mask.sum()
            
            print(f"\n{name_a} → {name_b}:")
            print(f"  Pairs with cosine sim >= {THRESHOLD}: {n_matches} / {len(emb_a)}")
            
            # Show top matches
            if n_matches > 0:
                top_idx = np.argsort(distances[:, 0])[::-1][:5]
                df_a = corpora[name_a]
                df_b = corpora[name_b]
                
                print(f"  Top matches:")
                for rank, idx_a in enumerate(top_idx):
                    idx_b = indices[idx_a, 0]
                    sim = distances[idx_a, 0]
                    
                    title_a = df_a.iloc[idx_a].get('title', df_a.iloc[idx_a].get('tune_name', 'N/A'))
                    title_b = df_b.iloc[idx_b].get('title', df_b.iloc[idx_b].get('tune_name', 'N/A'))
                    
                    print(f"    {rank+1}. sim={sim:.4f}")
                    print(f"       {name_a}: '{title_a}'")
                    print(f"       {name_b}: '{title_b}'")
else:
    print("Only one corpus available — skipping cross-corpus detection.")

## 13.4 Similarity Distribution & Threshold Calibration

In [ ]:
# Use the retrieval benchmark to calibrate a "same-tune" threshold
retrieval_full = pd.read_parquet(BENCHMARK_DIR / 'retrieval_full.parquet')

# Encode all retrieval tunes
full_emb = encode_tunes(retrieval_full['abc_body'].tolist(), model, patchifier, device)

# Normalize
fn = full_emb / np.maximum(np.linalg.norm(full_emb, axis=1, keepdims=True), 1e-8)

# Sample pairwise similarities
n = len(fn)
n_pairs = min(100000, n * (n-1) // 2)
np.random.seed(42)

pos_sims = []  # Same family
neg_sims = []  # Different family

families = retrieval_full['family_id'].values

# Sample pairs
for _ in range(n_pairs):
    i, j = np.random.choice(n, 2, replace=False)
    sim = np.dot(fn[i], fn[j])
    if families[i] == families[j]:
        pos_sims.append(sim)
    else:
        neg_sims.append(sim)

pos_sims = np.array(pos_sims)
neg_sims = np.array(neg_sims)

print(f"Same-family pairs: {len(pos_sims)}")
print(f"  Mean sim: {pos_sims.mean():.4f}, Std: {pos_sims.std():.4f}")
print(f"Different-family pairs: {len(neg_sims)}")
print(f"  Mean sim: {neg_sims.mean():.4f}, Std: {neg_sims.std():.4f}")

# Distribution plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(neg_sims, bins=100, alpha=0.6, label='Different tunes', color='blue', density=True)
ax.hist(pos_sims, bins=100, alpha=0.6, label='Same tune family', color='red', density=True)
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Density')
ax.set_title('Similarity Distribution: Same vs Different Tune Families')
ax.legend()

# Find optimal threshold (minimize overlap)
all_sims = np.concatenate([pos_sims, neg_sims])
all_labels = np.concatenate([np.ones(len(pos_sims)), np.zeros(len(neg_sims))])

precision, recall, thresholds = precision_recall_curve(all_labels, all_sims)
f1_scores = 2 * precision * recall / np.maximum(precision + recall, 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

ax.axvline(x=best_threshold, color='green', linestyle='--', linewidth=2,
           label=f'Optimal threshold: {best_threshold:.3f}')
ax.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plagiarism_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nOptimal 'same-tune' threshold: {best_threshold:.4f}")
print(f"  Precision: {precision[best_idx]:.4f}")
print(f"  Recall: {recall[best_idx]:.4f}")
print(f"  F1: {f1_scores[best_idx]:.4f}")

## 13.5 Precision-Recall Curve

In [ ]:
ap = average_precision_score(all_labels, all_sims)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall, precision, color='steelblue', linewidth=2)
ax.fill_between(recall, precision, alpha=0.2, color='steelblue')
ax.scatter([recall[best_idx]], [precision[best_idx]], color='red', s=100, 
           zorder=5, label=f'Best F1 (t={best_threshold:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'Melodic Similarity Detection — PR Curve (AP={ap:.4f})')
ax.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'plagiarism_pr_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 13.6 Bar-Level Alignment

In [ ]:
# For a pair of similar tunes, show which bars match
# This localizes the similarity to specific musical phrases

@torch.no_grad()
def get_bar_embeddings(abc_text, model, patchifier, device):
    """
    Get per-bar embeddings (from the transformer's hidden states).
    Returns: (n_bars, d_model) numpy array
    """
    model.eval()
    bars = patchifier.patchify(abc_text)
    ids, mask = patchifier.encode(bars)
    
    bar_ids = ids.unsqueeze(0).to(device)  # (1, max_bars, max_bar_len)
    bar_mask = mask.unsqueeze(0).to(device)  # (1, max_bars)
    
    # Get encoder hidden states (before tune projection)
    x = model.encoder.patch_embed(bar_ids)  # (1, max_bars, d_model)
    for layer in model.encoder.layers:
        x = layer(x, bar_mask)
    x = model.encoder.final_norm(x)  # (1, max_bars, d_model)
    
    # Mask out padding
    n_real_bars = mask.sum().item()
    bar_embs = x[0, :n_real_bars].cpu().numpy()  # (n_real_bars, d_model)
    
    return bar_embs, bars[:n_real_bars]

def plot_bar_alignment(abc_a, abc_b, model, patchifier, device):
    """Visualize bar-level similarity between two tunes."""
    emb_a, bars_a = get_bar_embeddings(abc_a, model, patchifier, device)
    emb_b, bars_b = get_bar_embeddings(abc_b, model, patchifier, device)
    
    # Normalize and compute similarity
    na = emb_a / np.maximum(np.linalg.norm(emb_a, axis=1, keepdims=True), 1e-8)
    nb = emb_b / np.maximum(np.linalg.norm(emb_b, axis=1, keepdims=True), 1e-8)
    
    sim_matrix = np.dot(na, nb.T)  # (n_bars_a, n_bars_b)
    
    fig, ax = plt.subplots(figsize=(max(8, len(bars_b)*0.5), max(6, len(bars_a)*0.4)))
    sns.heatmap(sim_matrix, cmap='RdYlBu_r', vmin=-0.5, vmax=1.0,
                xticklabels=[b[:20] for b in bars_b],
                yticklabels=[b[:20] for b in bars_a],
                ax=ax)
    ax.set_xlabel('Tune B (bars)')
    ax.set_ylabel('Tune A (bars)')
    ax.set_title('Bar-Level Cosine Similarity')
    
    plt.tight_layout()
    return fig, sim_matrix

# Example: show alignment for a known pair
if KNOWN_PAIRS:
    pair = KNOWN_PAIRS[0]  # The Kesh Jig variants
    print(f"Bar alignment for: {pair['name']}")
    fig, sim_matrix = plot_bar_alignment(
        pair['tune_a'], pair['tune_b'], model, patchifier, device
    )
    plt.savefig(RESULTS_DIR / 'plagiarism_bar_alignment.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Show which bars are most aligned
    for i in range(sim_matrix.shape[0]):
        best_j = np.argmax(sim_matrix[i])
        if sim_matrix[i, best_j] > 0.7:
            print(f"  Bar A[{i}] ↔ Bar B[{best_j}]: sim={sim_matrix[i, best_j]:.3f}")

## 13.7 Corpus-Level Similarity Heat Map

In [ ]:
# Average pairwise similarity between all pairs of corpora
if len(corpus_embeddings) >= 2:
    names = list(corpus_embeddings.keys())
    n_corpora = len(names)
    sim_matrix = np.zeros((n_corpora, n_corpora))
    
    for i in range(n_corpora):
        for j in range(n_corpora):
            emb_i = corpus_embeddings[names[i]]
            emb_j = corpus_embeddings[names[j]]
            
            # Normalize
            ni = emb_i / np.maximum(np.linalg.norm(emb_i, axis=1, keepdims=True), 1e-8)
            nj = emb_j / np.maximum(np.linalg.norm(emb_j, axis=1, keepdims=True), 1e-8)
            
            # Sample pairs for efficiency
            n_sample = min(500, len(ni), len(nj))
            idx_i = np.random.choice(len(ni), n_sample, replace=False)
            idx_j = np.random.choice(len(nj), n_sample, replace=False)
            
            sims = cosine_similarity(ni[idx_i], nj[idx_j])
            sim_matrix[i, j] = sims.mean()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(sim_matrix, annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=names, yticklabels=names, ax=ax)
    ax.set_title('Average Pairwise Similarity Between Corpora')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'plagiarism_corpus_similarity.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Need at least 2 corpora for cross-corpus analysis.")

## 13.8 Intra-Corpus Near-Duplicate Detection

In [ ]:
# Find near-duplicates within a single corpus
# These might be genuine variants, or actual duplicates in the dataset

main_corpus = 'Irish (IrishMAN)'
if main_corpus in corpus_embeddings:
    emb = corpus_embeddings[main_corpus]
    df = corpora[main_corpus]
    
    # Normalize
    en = emb / np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-8)
    
    # Build index and find nearest non-self neighbor
    d = emb.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(en.astype(np.float32))
    
    # Search for top-2 (first will be self)
    distances, indices = index.search(en.astype(np.float32), 2)
    
    # Nearest neighbor (excluding self)
    nn_sims = distances[:, 1]  # Second nearest = nearest non-self
    nn_idx = indices[:, 1]
    
    # Distribution of nearest-neighbor similarities
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(nn_sims, bins=100, color='steelblue', edgecolor='white')
    ax.axvline(x=best_threshold, color='red', linestyle='--', linewidth=2,
               label=f'Threshold: {best_threshold:.3f}')
    ax.set_xlabel('Cosine Similarity to Nearest Neighbor')
    ax.set_ylabel('Count')
    ax.set_title(f'Nearest-Neighbor Similarity Distribution ({main_corpus})')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'plagiarism_nn_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # List suspicious near-duplicates
    suspicious_mask = nn_sims >= best_threshold
    n_suspicious = suspicious_mask.sum()
    
    print(f"\nNear-duplicates above threshold ({best_threshold:.3f}): {n_suspicious} / {len(df)}")
    print(f"\nTop 10 most similar pairs:")
    top_pairs = np.argsort(nn_sims)[::-1][:10]
    for idx in top_pairs:
        nn = nn_idx[idx]
        sim = nn_sims[idx]
        title_a = df.iloc[idx].get('title', df.iloc[idx].get('tune_name', 'N/A'))
        title_b = df.iloc[nn].get('title', df.iloc[nn].get('tune_name', 'N/A'))
        print(f"  sim={sim:.4f}: '{title_a}' ↔ '{title_b}'")

## 13.9 Summary

In [ ]:
print("=" * 60)
print("Plagiarism / Melodic Similarity Detection — Summary")
print("=" * 60)
print(f"\nOptimal threshold: {best_threshold:.4f}")
print(f"  Precision: {precision[best_idx]:.4f}")
print(f"  Recall: {recall[best_idx]:.4f}")
print(f"  F1: {f1_scores[best_idx]:.4f}")
print(f"  AP: {ap:.4f}")
print(f"\nSame-family sim:    {pos_sims.mean():.4f} ± {pos_sims.std():.4f}")
print(f"Different-family:   {neg_sims.mean():.4f} ± {neg_sims.std():.4f}")
print(f"Separation:         {pos_sims.mean() - neg_sims.mean():.4f}")
print(f"\nApplications:")
print(f"  1. Detect tune variants across collections")
print(f"  2. Find cross-tradition borrowing")
print(f"  3. Bar-level similarity for localized comparison")
print(f"  4. Dataset deduplication")